# Axion PINN V4 — Physics-Informed Neural Network (Improved)

## Improvements over previous versions

| # | Improvement | Description |
|---|---|---|
| 1 | **Bug fix: density formula** | `rho_m0/a³`, `rho_r0/a⁴` — no double `a_eq` multiplication |
| 2 | **Hard IC for `a(t)`** | `a(t) = a₀ · exp(clamp(t·g(t), max=40))` — exact `a(0)=a₀`, overflow-safe |
| 3 | **Hard IC for `φ(t)`** | `φ(t) = φ₀ + φ̇₀·t + t²·NN(t)` — exact `φ(0)=φ₀`, `φ̇(0)=φ̇₀` |
| 4 | **Log-space `a(t)` + clamp** | Prevents overflow for large scale factor range |
| 5 | **Pretrain `a_net` on ODE** | Supervised in log-space before physics training |
| 6 | **Adaptive collocation** | 60% uniform + 40% near oscillation onset `t ~ 1/m` |
| 7 | **Separate Friedmann / KG weights** | `w_F` and `w_KG` independently tunable |
| 8 | **Gradient clipping** | Prevents gradient explosion for large `m` |
| 9 | **NaN recovery + best checkpoint** | Automatic restore + LR halving on NaN streak |
| 10 | **Consistent ODE ↔ PINN equations** | Both use `ȧ = aH`, `H² = ρ_tot/3`, standard KG |

### Equations of Motion (standard convention)
$$H = \frac{\dot{a}}{a} = \frac{1}{\sqrt{3}}\sqrt{\rho_\text{tot}}, \quad \rho_\text{tot} = \frac{1}{2}\dot\phi^2 + \frac{1}{2}m^2\phi^2 + \frac{\rho_{m0}}{a^3} + \frac{\rho_{r0}}{a^4} + \rho_\Lambda$$
$$\ddot\phi + 3H\dot\phi + m^2\phi = 0$$

In [6]:
import os
import time
import warnings
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.integrate import solve_ivp

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

PyTorch 2.12.0+cpu | CUDA: False


## Architecture

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. FourierFeatures  ─  t → [sin(Bt), cos(Bt)]
# ─────────────────────────────────────────────────────────────────────────────
class FourierFeatures(nn.Module):
    """Random Fourier Features. Maps scalar t → 2*out_features dimensional vector."""
    def __init__(self, out_features=64, scale=10.0, device=None, dtype=torch.float32):
        super().__init__()
        B = torch.randn(out_features, 1, device=device, dtype=dtype) * scale
        self.register_buffer('B', B)

    def forward(self, t):
        proj = 2.0 * np.pi * (t @ self.B.T)          # (N, out_features)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)   # (N, 2*out_features)


# ─────────────────────────────────────────────────────────────────────────────
# 2. ResidualBlock
# ─────────────────────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.fc1 = nn.Linear(width, width)
        self.fc2 = nn.Linear(width, width)

    def forward(self, x):
        return x + self.fc2(torch.sin(self.fc1(x)))


# ─────────────────────────────────────────────────────────────────────────────
# 3. LogScaleNet  ─  Hard IC for a(t)
#    a(t) = a0 * exp( clamp(t * g(t), max=40) )
#    → a(0) = a0 * exp(0) = a0  (exact)
#    → clamping prevents overflow when a grows by many orders of magnitude
# ─────────────────────────────────────────────────────────────────────────────
class LogScaleNet(nn.Module):
    def __init__(self, a0, hidden=64, depth=4, device=None, dtype=torch.float32):
        super().__init__()
        self.register_buffer('a0', torch.tensor([[float(a0)]], dtype=dtype))
        layers = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.Tanh()]
        layers.append(nn.Linear(hidden, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        g = self.net(t)                          # (N, 1) — learned rate
        exponent = (t * g).clamp(max=40.0)       # hard clamp: no overflow
        return self.a0 * torch.exp(exponent)     # a(0) = a0  (exact IC)


# ─────────────────────────────────────────────────────────────────────────────
# 4. HardICPhiNet  ─  Hard IC for φ(t)
#    φ(t) = φ0 + φ̇0*t + t² * NN(t)
#    → φ(0) = φ0        (exact)
#    → dφ/dt(0) = φ̇0   (exact)
# ─────────────────────────────────────────────────────────────────────────────
class HardICPhiNet(nn.Module):
    def __init__(self, N_fields, phi0, phi_dot0,
                 embed_dim=64, hidden=128, blocks=3, fourier_scale=10.0,
                 device=None, dtype=torch.float32):
        super().__init__()
        self.N_fields = N_fields
        self.register_buffer(
            'phi0',
            torch.tensor(np.asarray(phi0, dtype=np.float32), dtype=dtype, device=device).reshape(1, -1)
        )
        self.register_buffer(
            'phi_dot0',
            torch.tensor(np.asarray(phi_dot0, dtype=np.float32), dtype=dtype, device=device).reshape(1, -1)
        )
        self.fourier = FourierFeatures(embed_dim, fourier_scale, device=device, dtype=dtype)
        layers = [nn.Linear(2 * embed_dim, hidden)]
        for _ in range(blocks):
            layers.append(ResidualBlock(hidden))
        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(hidden, N_fields)

    def forward(self, t):
        phi_nn = self.out(self.net(self.fourier(t)))     # residual correction
        return self.phi0 + self.phi_dot0 * t + t**2 * phi_nn


# ─────────────────────────────────────────────────────────────────────────────
# 5. PINN_V4  ─  Combined model
# ─────────────────────────────────────────────────────────────────────────────
class PINN_V4(nn.Module):
    def __init__(self, N_fields, a0, phi0, phi_dot0,
                 embed_dim=64, hidden=128, blocks=3, fourier_scale=10.0,
                 device=None, dtype=torch.float32):
        super().__init__()
        self.a_net   = LogScaleNet(a0, hidden=hidden // 2, device=device, dtype=dtype)
        self.phi_net = HardICPhiNet(N_fields, phi0, phi_dot0,
                                    embed_dim, hidden, blocks, fourier_scale, device, dtype)

    def forward(self, t):
        return self.a_net(t), self.phi_net(t)


print("Architecture classes defined: FourierFeatures, ResidualBlock, LogScaleNet, HardICPhiNet, PINN_V4")

Architecture classes defined: FourierFeatures, ResidualBlock, LogScaleNet, HardICPhiNet, PINN_V4


## PINNSolver V4

In [3]:
class PINNSolver_V4:
    """
    Physics-Informed Neural Network solver for Axion Cosmology (V4).

    Equations (standard flat FLRW):
        Friedmann : ȧ = a·H,  H = sqrt(ρ_tot / 3)
        Klein-Gordon: φ̈ + 3H·φ̇ + m²·φ = 0
        ρ_tot = ½φ̇² + ½m²φ² + ρ_m0/a³ + ρ_r0/a⁴ + ρ_Λ

    Parameters (densities defined at a=1 today):
        rho_m0 : matter density
        rho_r0 : radiation density
        rho_l0 : cosmological constant (dark energy)
    """

    def __init__(self, N_fields=1, m_vec=None,
                 rho_m0=0.81, rho_r0=0.00027138, rho_l0=2.19,
                 a0=1e-8, phi0=None, phi_dot0=None,
                 t_span=(1e-10, 1.0), n_eval=5000,
                 folder='results_v4', device=None, dtype=torch.float32):

        self.device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        self.dtype = dtype
        self.folder = folder
        os.makedirs(folder, exist_ok=True)
        print(f"Device: {self.device}")

        # Physics parameters
        self.N = int(N_fields)
        self.m_vec = np.array(m_vec if m_vec else [100.0] * self.N, dtype=np.float64)
        self.rho_m0 = float(rho_m0)   # density at a=1
        self.rho_r0 = float(rho_r0)
        self.rho_l0 = float(rho_l0)

        # Initial conditions
        self.a0      = float(a0)
        self.phi0    = np.array(phi0 if phi0 else [1.0] * self.N, dtype=np.float64)
        self.phi_dot0= np.array(phi_dot0 if phi_dot0 else [0.0] * self.N, dtype=np.float64)
        self.y0      = np.concatenate([[self.a0], self.phi0, self.phi_dot0])
        self.t_span  = t_span

        # Log-spaced evaluation grid (covers early times well)
        t_lo = max(float(t_span[0]), 1e-12)
        t_hi = float(t_span[1])
        self.t_eval = np.logspace(np.log10(t_lo), np.log10(t_hi), n_eval, dtype=np.float32)

        # Build model
        self.model = PINN_V4(
            self.N, self.a0, self.phi0, self.phi_dot0,
            device=self.device, dtype=self.dtype
        ).to(self.device, self.dtype)

        # Torch constants
        self.m_torch = torch.tensor(self.m_vec, dtype=dtype, device=self.device).reshape(1, -1)
        self.EPS = 1e-30   # numerical safety floor

        # NaN-recovery state
        self._best_loss  = float('inf')
        self._best_state = None
        self._nan_streak = 0

    # ──────────────────────────────────────────────
    # ODE reference  (same equations as PINN)
    # ──────────────────────────────────────────────
    def _ode(self, t, y):
        N = self.N
        a      = y[0]
        phi    = y[1:N+1]
        phi_dot= y[N+1:2*N+1]

        rho = (0.5 * np.sum(phi_dot**2)
               + 0.5 * np.sum(self.m_vec**2 * phi**2)
               + self.rho_m0 / a**3
               + self.rho_r0 / a**4
               + self.rho_l0)
        H = np.sqrt(max(rho, 0.0) / 3.0)

        dy = np.empty_like(y)
        dy[0]       = a * H                              # ȧ = a·H
        dy[1:N+1]   = phi_dot
        dy[N+1:2*N+1] = -3.0 * H * phi_dot - self.m_vec**2 * phi   # KG
        return dy

    def solve_ode(self, rtol=1e-8, atol=1e-11):
        print("Solving ODE reference (RK45)...")
        t0 = time.time()
        sol = solve_ivp(self._ode, self.t_span, self.y0,
                        t_eval=self.t_eval, method='RK45', rtol=rtol, atol=atol)
        dt = time.time() - t0
        if not sol.success:
            warnings.warn(f"ODE solver warning: {sol.message}")
        print(f"  Done in {dt:.2f}s | success={sol.success}")
        self.a_sol    = sol.y[0].astype(np.float32)
        self.phi_sol  = sol.y[1:1+self.N].astype(np.float32)   # shape (N, T)
        self.dphi_sol = sol.y[1+self.N:1+2*self.N].astype(np.float32)

    # ──────────────────────────────────────────────
    # Pretrain a_net on ODE solution (log-space)
    # ──────────────────────────────────────────────
    def pretrain_a_net(self, n_epochs=5000, lr=1e-3, print_every=1000):
        """
        Supervised pretraining of a_net using ODE solution in log-space.
        Loss = MSE(log(a_pred), log(a_ode)) — robust across many orders of magnitude.
        """
        if not hasattr(self, 'a_sol'):
            raise RuntimeError("Call solve_ode() first.")

        t_data  = torch.tensor(self.t_eval,  dtype=self.dtype, device=self.device).reshape(-1, 1)
        a_data  = torch.tensor(self.a_sol,   dtype=self.dtype, device=self.device).reshape(-1, 1)

        opt = optim.Adam(self.model.a_net.parameters(), lr=lr)
        sch = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=500, factor=0.5, min_lr=1e-7)

        best, t0 = float('inf'), time.time()
        print(f"Pretraining a_net ({n_epochs} epochs, log-MSE)...")
        for ep in range(n_epochs):
            opt.zero_grad()
            a_pred = self.model.a_net(t_data)
            # log-space loss: handles 8+ orders of magnitude in a(t)
            loss = torch.mean(
                (torch.log(a_pred + self.EPS) - torch.log(a_data + self.EPS))**2
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.a_net.parameters(), 1.0)
            opt.step()
            sch.step(loss)
            lv = loss.item()
            if lv < best:
                best = lv
            if ep % print_every == 0 or ep == n_epochs - 1:
                print(f"  ep {ep:5d}/{n_epochs} | log-MSE={lv:.3e} | best={best:.3e}")
        print(f"  Pretrain done in {time.time()-t0:.1f}s | best log-MSE={best:.3e}")

    # ──────────────────────────────────────────────
    # Physics loss  (Friedmann + Klein-Gordon)
    # ──────────────────────────────────────────────
    def _physics_loss(self, t_b, w_F=1.0, w_KG=1.0):
        a, phi = self.model(t_b)

        a_t    = torch.autograd.grad(a,    t_b, torch.ones_like(a),    create_graph=True)[0]
        phi_t  = torch.autograd.grad(phi,  t_b, torch.ones_like(phi),  create_graph=True)[0]
        phi_tt = torch.autograd.grad(phi_t, t_b, torch.ones_like(phi_t), create_graph=True)[0]

        # ── Densities (correct: ρ_m0 at a=1, NO double a_eq factor) ──
        rho_ax = (0.5 * torch.sum(phi_t**2,                   dim=1, keepdim=True) +
                  0.5 * torch.sum(self.m_torch**2 * phi**2,   dim=1, keepdim=True))
        rho_m  = self.rho_m0 / (a**3 + self.EPS)
        rho_r  = self.rho_r0 / (a**4 + self.EPS)
        rho_tot = rho_ax + rho_m + rho_r + self.rho_l0

        H = torch.sqrt(torch.clamp(rho_tot / 3.0, min=0.0) + self.EPS)

        # ── Residuals ──
        R_F  = a_t - a * H                                       # Friedmann
        R_KG = phi_tt + 3.0 * H * phi_t + self.m_torch**2 * phi # Klein-Gordon

        L_F  = torch.mean(R_F**2)
        L_KG = torch.mean(R_KG**2)
        return w_F * L_F + w_KG * L_KG, L_F.detach().item(), L_KG.detach().item()

    # ──────────────────────────────────────────────
    # Adaptive collocation sampling
    # 60% uniform in [t_min, t_max]
    # 40% concentrated near oscillation onset t ~ 1/m
    # ──────────────────────────────────────────────
    def _collocation(self, N_f):
        t_lo = max(float(self.t_span[0]), 1e-10)
        t_hi = float(self.t_span[1])
        n_u = int(0.6 * N_f)
        n_o = N_f - n_u

        t_u = torch.rand(n_u, 1, device=self.device, dtype=self.dtype) * (t_hi - t_lo) + t_lo

        # Oscillation onset: m·t ~ 1  →  t_osc ≈ 1/m_max
        t_osc  = min(1.0 / float(self.m_vec.max()), t_hi * 0.5)
        sigma  = t_osc * 0.4
        t_near = (torch.randn(n_o, 1, device=self.device, dtype=self.dtype) * sigma + t_osc
                  ).clamp(t_lo, t_hi)

        return torch.cat([t_u, t_near], dim=0)

    # ──────────────────────────────────────────────
    # Checkpoint helpers
    # ──────────────────────────────────────────────
    def _save_ckpt(self):
        self._best_state = deepcopy(self.model.state_dict())

    def _restore_ckpt(self):
        if self._best_state is not None:
            self.model.load_state_dict(self._best_state)
            print("  ↩ Restored best checkpoint")

    # ──────────────────────────────────────────────
    # Adam training
    # ──────────────────────────────────────────────
    def train(self, epochs=10000, w_F=1.0, w_KG=1.0,
              N_f=2000, batch=512, lr=1e-3, grad_clip=1.0, print_every=1000):
        opt = optim.Adam(self.model.parameters(), lr=lr)
        # Cosine LR decay: smooth warmup → cooldown
        sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)

        self.hist = {'total': [], 'F': [], 'KG': []}
        print(f"\nAdam training ({epochs} epochs | w_F={w_F} w_KG={w_KG} | batch={batch})...")
        t0 = time.time()

        for ep in range(epochs):
            t_all = self._collocation(N_f)
            idx   = torch.randperm(N_f, device=self.device)[:batch]
            t_b   = t_all[idx].requires_grad_(True)

            opt.zero_grad()
            L, Lf, Lkg = self._physics_loss(t_b, w_F, w_KG)
            L.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), grad_clip)  # gradient clipping
            opt.step()
            sch.step()

            lv = L.item()

            # ── NaN recovery ──────────────────────────────────────────────
            if np.isnan(lv) or np.isinf(lv):
                self._nan_streak += 1
                if self._nan_streak >= 5:
                    print(f"  ⚠ NaN streak={self._nan_streak}: restoring checkpoint + halving LR")
                    self._restore_ckpt()
                    for pg in opt.param_groups:
                        pg['lr'] *= 0.5
                    self._nan_streak = 0
                continue
            self._nan_streak = 0

            # ── Save best ─────────────────────────────────────────────────
            if lv < self._best_loss:
                self._best_loss = lv
                self._save_ckpt()

            self.hist['total'].append(lv)
            self.hist['F'].append(Lf)
            self.hist['KG'].append(Lkg)

            if ep % print_every == 0 or ep == epochs - 1:
                pct    = 100 * ep / epochs
                lr_now = sch.get_last_lr()[0]
                print(f"  ep {ep:6d}/{epochs} ({pct:5.1f}%) "
                      f"| L={lv:.3e}  F={Lf:.3e}  KG={Lkg:.3e}  lr={lr_now:.2e}")

        print(f"Adam done in {time.time()-t0:.1f}s | best_loss={self._best_loss:.3e}")
        self._restore_ckpt()

    # ──────────────────────────────────────────────
    # L-BFGS refinement
    # ──────────────────────────────────────────────
    def lbfgs(self, w_F=1.0, w_KG=1.0, N_f=2000, max_iter=2000):
        print("\nL-BFGS refinement...")
        t_f = self._collocation(N_f).requires_grad_(True)
        opt = optim.LBFGS(self.model.parameters(), lr=1.0, max_iter=max_iter,
                           tolerance_grad=1e-9, tolerance_change=1e-11,
                           history_size=100, line_search_fn='strong_wolfe')
        it = [0]

        def closure():
            opt.zero_grad()
            L, Lf, Lkg = self._physics_loss(t_f, w_F, w_KG)
            L.backward()
            it[0] += 1
            if it[0] % 500 == 0:
                print(f"  L-BFGS iter {it[0]:4d} | L={L.item():.3e}")
            return L

        t0 = time.time()
        opt.step(closure)
        print(f"L-BFGS done in {time.time()-t0:.1f}s | iters={it[0]}")


print("PINNSolver_V4 defined.")

PINNSolver_V4 defined.


## Evaluation & Plotting

In [4]:
def evaluate_and_plot(solver, title_suffix=""):
    """
    Evaluate model on t_eval, compare against ODE, and produce plots.
    Also prints relative error metrics.
    """
    solver.model.eval()
    with torch.no_grad():
        t_t = torch.tensor(
            solver.t_eval, dtype=solver.dtype, device=solver.device
        ).reshape(-1, 1)
        a_t, phi_t = solver.model(t_t)
    a_np   = a_t.cpu().numpy().flatten()
    phi_np = phi_t.cpu().numpy().T    # (N_fields, T)

    # ── NaN guard ──────────────────────────────────────────────────────────
    if np.isnan(a_np).any():
        n = np.isnan(a_np).sum()
        warnings.warn(f"⚠ {n} NaN in a_pred — possible overflow during training")
        a_np = np.nan_to_num(a_np)
    if np.isnan(phi_np).any():
        n = np.isnan(phi_np).sum()
        warnings.warn(f"⚠ {n} NaN in phi_pred")
        phi_np = np.nan_to_num(phi_np)

    solver.a_pred   = a_np
    solver.phi_pred = phi_np

    t  = solver.t_eval
    N  = solver.N

    # ── IC verification ────────────────────────────────────────────────────
    print("=== IC Verification ===")
    print(f"  a(t_min)  : pred={a_np[0]:.6e}  ODE={solver.a_sol[0]:.6e}  "
          f"true_a0={solver.a0:.6e}")
    for i in range(N):
        print(f"  φ_{i}(t_min): pred={phi_np[i,0]:.6e}  ODE={solver.phi_sol[i,0]:.6e}  "
              f"true_phi0={solver.phi0[i]:.6e}")

    # ── Plot a(t) and φ(t) ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

    ax = axes[0]
    ax.loglog(t, solver.a_sol, 'k--', lw=2,   label='ODE a(t)')
    ax.loglog(t, a_np,         'r-',  lw=1.5,  label='PINN V4 a(t)')
    ax.set(xlabel='t', ylabel='a(t)', title=f'Scale Factor{title_suffix}')
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    colors = plt.cm.tab10(np.linspace(0, 0.45, max(min(N, 4), 1)))
    for i in range(min(N, 4)):
        ax.semilogx(t, solver.phi_sol[i], 'k--',         lw=2,   alpha=0.6, label=f'ODE φ_{i}')
        ax.semilogx(t, phi_np[i],         color=colors[i], lw=1.5, label=f'PINN φ_{i}')
    ax.set(xlabel='t', ylabel='φ(t)', title=f'Axion Field{title_suffix}')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(solver.folder, 'comparison_v4.png'), dpi=120, bbox_inches='tight')
    plt.show()

    # ── Residual plots (|PINN − ODE| / |ODE|) ─────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 4), dpi=120)
    eps = 1e-30

    ax = axes[0]
    rel_a = np.abs(a_np - solver.a_sol) / (np.abs(solver.a_sol) + eps)
    ax.loglog(t, rel_a, 'r-', lw=1.2)
    ax.set(xlabel='t', ylabel='|pred−ODE|/|ODE|', title=f'Relative Error a(t){title_suffix}')
    ax.grid(True, alpha=0.3)

    ax = axes[1]
    for i in range(min(N, 4)):
        scale = np.abs(solver.phi_sol[i]).max() + eps
        rel_p = np.abs(phi_np[i] - solver.phi_sol[i]) / scale
        ax.loglog(t, rel_p, lw=1.2, label=f'φ_{i}')
    ax.set(xlabel='t', ylabel='|pred−ODE|/max|ODE|', title=f'Relative Error φ(t){title_suffix}')
    ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(solver.folder, 'residuals_v4.png'), dpi=120, bbox_inches='tight')
    plt.show()

    # ── Loss history ───────────────────────────────────────────────────────
    if hasattr(solver, 'hist') and solver.hist['total']:
        plt.figure(figsize=(10, 4), dpi=120)
        plt.semilogy(solver.hist['total'], 'b-',  lw=1.5, alpha=0.9, label='Total')
        plt.semilogy(solver.hist['F'],     'g--', lw=1.0, alpha=0.8, label='Friedmann')
        plt.semilogy(solver.hist['KG'],    'r--', lw=1.0, alpha=0.8, label='Klein-Gordon')
        plt.xlabel('Epoch'); plt.ylabel('Loss')
        plt.title(f'Training Loss V4{title_suffix}')
        plt.legend(); plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(solver.folder, 'loss_v4.png'), dpi=120, bbox_inches='tight')
        plt.show()

    # ── Error metrics ──────────────────────────────────────────────────────
    print("\n=== Error Metrics ===")
    err_a = np.abs(a_np - solver.a_sol) / (np.abs(solver.a_sol) + eps)
    print(f"  a(t)  : max_rel={err_a.max():.3e} | mean_rel={err_a.mean():.3e}")
    for i in range(N):
        scale  = np.abs(solver.phi_sol[i]).max() + eps
        err_p  = np.abs(phi_np[i] - solver.phi_sol[i]) / scale
        print(f"  φ_{i}(t): max_rel={err_p.max():.3e} | mean_rel={err_p.mean():.3e}")


print("evaluate_and_plot defined.")

evaluate_and_plot defined.


## Run — Single Field (N=1, m=70)

In [5]:
if __name__ == "__main__":
    t_total = time.time()

    # ── Build solver ──────────────────────────────────────────────────────
    s = PINNSolver_V4(
        N_fields=1,
        m_vec=[70],
        rho_m0=0.81,
        rho_r0=0.00027138,
        rho_l0=2.19,
        a0=1e-8,
        phi0=[1.0],
        phi_dot0=[0.0],
        t_span=(1e-10, 1.0),
        n_eval=5000,
        folder='results_v4',
    )

    # ── Step 1: ODE reference ─────────────────────────────────────────────
    s.solve_ode()

    # ── Step 2: Pretrain a_net on ODE data ────────────────────────────────
    s.pretrain_a_net(n_epochs=5000, lr=1e-3, print_every=1000)

    # ── Step 3: Adam physics training ────────────────────────────────────
    s.train(
        epochs=10000,
        w_F=1.0, w_KG=1.0,
        N_f=2000, batch=512,
        lr=1e-3, grad_clip=1.0,
        print_every=1000,
    )

    # ── Step 4: L-BFGS refinement ────────────────────────────────────────
    s.lbfgs(w_F=1.0, w_KG=1.0, N_f=2000, max_iter=2000)

    print(f"\nTotal pipeline time: {time.time()-t_total:.1f}s")

Device: cpu
Solving ODE reference (RK45)...
  Done in 0.31s | success=True
Pretraining a_net (5000 epochs, log-MSE)...
  ep     0/5000 | log-MSE=1.401e+02 | best=1.401e+02


c:\Users\771754\Documents\Works\New_ver_PINNs_axion\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:1691: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:839.)
  current = float(metrics)


  ep  1000/5000 | log-MSE=1.039e+02 | best=1.039e+02
  ep  2000/5000 | log-MSE=9.472e+01 | best=9.472e+01
  ep  3000/5000 | log-MSE=9.066e+01 | best=9.066e+01
  ep  4000/5000 | log-MSE=8.833e+01 | best=8.833e+01
  ep  4999/5000 | log-MSE=8.616e+01 | best=8.616e+01
  Pretrain done in 35.1s | best log-MSE=8.616e+01

Adam training (10000 epochs | w_F=1.0 w_KG=1.0 | batch=512)...
  ep      0/10000 (  0.0%) | L=3.805e+19  F=1.529e+09  KG=3.805e+19  lr=1.00e-03
  ⚠ NaN streak=5: restoring checkpoint + halving LR
  ↩ Restored best checkpoint
  ⚠ NaN streak=5: restoring checkpoint + halving LR
  ↩ Restored best checkpoint
  ⚠ NaN streak=5: restoring checkpoint + halving LR
  ↩ Restored best checkpoint
  ⚠ NaN streak=5: restoring checkpoint + halving LR
  ↩ Restored best checkpoint
  ⚠ NaN streak=5: restoring checkpoint + halving LR
  ↩ Restored best checkpoint
  ⚠ NaN streak=5: restoring checkpoint + halving LR
  ↩ Restored best checkpoint
  ⚠ NaN streak=5: restoring checkpoint + halving LR
  

KeyboardInterrupt: 

In [ ]:
evaluate_and_plot(s, title_suffix=" (N=1, m=70)")

---
## (Optional) VoP (Variation of Parameters) Ansatz — for high-m

For $m \gg H$ (oscillating regime, e.g. $m \geq 100$), the direct network struggles to learn rapid oscillations.
The **VoP ansatz** decomposes:

$$\phi(t) = C_1(t)\cdot\underbrace{\frac{\sin(mt)}{\sqrt{mt+\varepsilon}}}_{j_{1/2}\text{-like}} + C_2(t)\cdot\underbrace{\frac{\cos(mt)}{\sqrt{mt+\varepsilon}}}_{y_{1/2}\text{-like}}$$

The network only needs to learn the **slowly-varying envelopes** $C_1(t)$ and $C_2(t)$, which is much easier.

> **Note:** This ansatz has a singularity at $t=0$ from the cosine term. Use with soft ICs or start from $t > 0$.
> For the exact matter-domination solution: $\phi = a^{-3/2}\sqrt{t/t_i}[C_1 J_{1/2}(mt) + C_2 Y_{1/2}(mt)]$

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# VoP φ-network:  φ(t) = C1(t)·sin(mt)/√(mt+ε) + C2(t)·cos(mt)/√(mt+ε)
# Network learns slowly-varying C1(t), C2(t) only.
# For high-m (m ≥ 100) this converges much faster than the direct approach.
# ─────────────────────────────────────────────────────────────────────────────
class VoPPhiNet(nn.Module):
    def __init__(self, N_fields, m_vec,
                 embed_dim=32, hidden=64, blocks=2, fourier_scale=1.0,
                 device=None, dtype=torch.float32):
        super().__init__()
        self.N = N_fields
        self.register_buffer('m', torch.tensor(m_vec, dtype=dtype).reshape(1, -1))
        # Low-frequency Fourier features for slowly-varying envelopes
        self.fourier = FourierFeatures(embed_dim, fourier_scale, device=device, dtype=dtype)
        layers = [nn.Linear(2 * embed_dim, hidden)]
        for _ in range(blocks):
            layers.append(ResidualBlock(hidden))
        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(hidden, 2 * N_fields)   # outputs C1 and C2 for each field

    def forward(self, t):
        C = self.out(self.net(self.fourier(t)))        # (B, 2N)
        C1 = C[:, :self.N]                             # (B, N)
        C2 = C[:, self.N:]                             # (B, N)
        mt      = self.m * t                           # (B, N)
        mt_safe = torch.clamp(mt, min=1e-30)
        inv_sqrt_mt = 1.0 / torch.sqrt(mt_safe)       # Bessel normalisation
        phi = C1 * torch.sin(mt) * inv_sqrt_mt + C2 * torch.cos(mt) * inv_sqrt_mt
        return phi


class PINN_V4_VoP(nn.Module):
    """PINN_V4 variant with VoP ansatz for φ — better for m ≥ 100."""
    def __init__(self, N_fields, a0, m_vec,
                 embed_dim=32, hidden=128, blocks=3, fourier_scale=1.0,
                 device=None, dtype=torch.float32):
        super().__init__()
        self.a_net   = LogScaleNet(a0, hidden=hidden // 2, device=device, dtype=dtype)
        self.phi_net = VoPPhiNet(N_fields, m_vec, embed_dim, hidden // 2, blocks,
                                  fourier_scale, device, dtype)

    def forward(self, t):
        return self.a_net(t), self.phi_net(t)


# ── Quick test: compare both architectures for m=200 ─────────────────────────
def run_vop_demo(m=200, epochs_adam=5000, epochs_lbfgs=1000):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype  = torch.float32

    # Shared ODE solver for reference
    solver_ref = PINNSolver_V4(
        N_fields=1, m_vec=[m],
        a0=1e-8, phi0=[1.0], phi_dot0=[0.0],
        t_span=(1e-10, 1.0), n_eval=3000,
        folder=f'results_v4_vop_m{m}',
    )
    solver_ref.solve_ode()

    # Build VoP model
    model_vop = PINN_V4_VoP(
        N_fields=1, a0=1e-8, m_vec=[m],
        embed_dim=32, hidden=128, blocks=3, fourier_scale=1.0,
        device=device, dtype=dtype,
    ).to(device, dtype)

    solver_ref.model = model_vop   # swap the model in the same solver
    solver_ref._best_loss  = float('inf')
    solver_ref._best_state = None
    solver_ref._nan_streak = 0

    # Pretrain a_net
    solver_ref.pretrain_a_net(n_epochs=3000, lr=1e-3, print_every=1000)

    # Physics training
    solver_ref.train(epochs=epochs_adam, w_F=1.0, w_KG=1.0,
                     N_f=2000, batch=512, lr=1e-3, print_every=1000)
    solver_ref.lbfgs(N_f=2000, max_iter=epochs_lbfgs)
    evaluate_and_plot(solver_ref, title_suffix=f" VoP (m={m})")


# Uncomment to run:
# run_vop_demo(m=200, epochs_adam=5000)

print("VoP classes defined: VoPPhiNet, PINN_V4_VoP, run_vop_demo()")